#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key" : "product_key",
    "prd_nm" : "product_name",
    "prd_cost" : "product_cost",
    "prd_line" : "product_line",
    "prd_start_dt" : "product_valid_from",
    "prd_end_dt" : "product_valid_to"
}

#Reading From Bronze

In [0]:
df = spark.table("workspace.bronze.crm_prod_info")

# Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
 


## Standardize prd_key, category_id

In [0]:
df = df.withColumn("category_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

## Null handling and Empty values

In [0]:
def is_empty(col):
    return col.isNull() | (F.trim(col) == "")

condition_invalid = (
    is_empty(F.col("prd_id")) |
    is_empty(F.col("prd_key")) |
    is_empty(F.col("prd_nm")) 
)

df_valid = df.filter(~condition_invalid)
df_invalid = df.filter(condition_invalid)

print("Total rows:", df.count())
print("Invalid rows:", df_invalid.count())
print("Valid rows:", df_valid.count())

df = df_valid

## Data type casting

In [0]:
df = (
    df.withColumn("prd_start_dt", F.try_to_date("prd_start_dt"))
      .withColumn("prd_end_dt", F.try_to_date("prd_end_dt"))
)


## Handle duplicate ids

In [0]:
duplicate_ids_count = (
    df.groupBy("prd_id")
      .count()
      .filter("count > 1")
      .count()
)

print("Duplicate prd_ids:", duplicate_ids_count)

if duplicate_ids_count > 0:
    raise Exception("Duplicate product IDs found!")

## Cost validation

In [0]:
invalid_cost_condition = (
    F.col("prd_cost").isNull() |
    (F.col("prd_cost") < 0)
)

df_invalid = df.filter(invalid_cost_condition)
df_valid = df.filter(~invalid_cost_condition)

invalid_count = df_invalid.count()
total_count = df.count()

print("Total rows:", total_count)
print("Invalid cost rows:", invalid_count)
print("Valid rows:", total_count - invalid_count)

df = df_valid

##Normalization

In [0]:
df = df.withColumn("prd_line",
                   F.when(F.upper(F.col("prd_line")) == "R", "Road")
                   .when(F.upper(F.col("prd_line")) == "S", "Sport")
                   .when(F.upper(F.col("prd_line")) == "M", "Mountain")
                   .when(F.upper(F.col("prd_line")) == "T", "Touring")
                   .otherwise("Unknown"))


## Renamig the columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)


## Sanity checks before write

In [0]:
def sanity_check(df):
    row_count = df.count()

    duplicate_product_id = (
        df.groupBy("product_id")
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    null_critical_fields = (
        df.filter(
            F.col("product_id").isNull() |
            F.col("product_key").isNull() |
            F.col("product_name").isNull() |
            F.col("product_valid_from").isNull()
        )
        .count()
    )

    invalid_cost = (
        df.filter(F.col("product_cost") < 0)
        .count()
    )

    return {
        "row_count": row_count,
        "duplicate_product_id": duplicate_product_id,
        "null_critical_fields": null_critical_fields,
        "invalid_cost": invalid_cost,
    }

results = sanity_check(df)
print("Before write:", results)


# Write Into Silver

In [0]:
(df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver.crm_products"))

df_silver = spark.table("silver.crm_products")

results = sanity_check(df_silver)
print("After write:", results)

if results["duplicate_product_id"] > 0:
    raise Exception("Duplicate product_id found!")

if results["null_critical_fields"] > 0:
    raise Exception("Null critical fields found!")

if results["invalid_cost"] > 0:
    raise Exception("Invalid cost values found!")

spark.table("silver.crm_products").display()